### Main Goal: Learning Google Cloud

#### Creating second classification model using BIGQUERY-ML and KUBEFLOW PIPELINE and replace the original model if the second is better.
 - Source Data: Project BigQuery Dataset table that we have created in last tutorial.
 - create a pipeline 
   - that trains a DNN model on BigQuey ML
   - Evaluate performance of both DNN and logistic regression that is already deployed.
   - Replace old model with new model in case if new performs better.

In [ ]:
!pip install kfp
!pip install google-cloud-storage
!pip install google-cloud-bigquery
!pip install google-cloud-aiplatform

In [5]:
from google.cloud import storage
from google.cloud import bigquery
from google.cloud import aiplatform

import kfp # used for dsl.pipeline
import kfp.dsl as dsl # used for dsl.component, dsl.Output, dsl.Input, dsl.Artifact, dsl.Model, ...

from datetime import datetime

In [6]:
# Project, experiment related variables
REGION = 'us-central1'
EXPERIMENT = 'pipeline_ex2'
SERIES = 'bqml'
PROJECT_ID = 'projectmail-431520'

# source data, bigquery variables
BQ_PROJECT = PROJECT_ID
BQ_DATASET = 'fraud'
BQ_TABLE = 'fraud_data'
BQ_SOURCE = f'{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}'

# Cloud storage variables
BUCKET = PROJECT_ID
URI = f"gs://{BUCKET}/{SERIES}/{EXPERIMENT}/pipelines"

# Model Training
VAR_TARGET = 'Class'
VAR_OMIT = 'transaction_id' # add more variables to the string with space delimiters

TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
RUN_NAME = f'run-{TIMESTAMP}'

# Create a directory for keeping temorary files
DIR = f"temp/{EXPERIMENT}"
!rm -rf {DIR}
!mkdir -p {DIR}

In [10]:
aiplatform.init(location=REGION, project=PROJECT_ID)
#bq = bigquery.Client(project=PROJECT_ID)
#gcs = storage.Client(project=PROJECT_ID)


### Custom Components (KFP)
Vertex AI Pipelines are made up of components that run independently with inputs and outputs that connect to form a graph - the pipeline. 
For this notebook workflow the following custom components are used to orchestrate the training of a challenger model, evaluating the challenger and an existing model, 
comparing them based on model metrics, if the challenger is better then replace the model already deployed on an existing endpoint. These custom components are constructed as python functions!

#### Get the Deployed Model

In [ ]:
@dsl.component(
    base_image="python:3.9",
    packages_to_install = ['google-cloud-aiplatform']
)
def get_deployed_model(
    project: str,
    region: str,
    series: str,
    bqml_model: dsl.Output[dsl.Artifact],
    vertex_endpoint: dsl.Output[dsl.Artifact]
):
    # We assume that there is already a model exists deployed on a endpoint. Not covering for no-model case
    from google.cloud import aiplatform
    aiplatform.init(project = project, location = region)
    
    # Get the Endpoint
    endpoint_list = aiplatform.Endpoint.list(filter=f"display_name={series}")
    if endpoint_list:
        endpoint = endpoint_list[0]
        print("Endpoint Exists", endpoint)
    else:
        print("No Endpoint Exists")

    # Get the model deployed
    print(endpoint.resource_name)
    print(endpoint.traffic_split)
    deployed_models = endpoint.list_models()
    if deployed_models:
        model = deployed_models[0]
        print(model.id)
        print(model.model)
        print(model.model_version_id)
        deployed_model = model.model+f'@{model.model_version_id}'
        print(deployed_model)
        deployed_model = aiplatform.Model(model_name = deployed_model)
        bq_model = deployed_model.display_name
        print(deployed_model)
        print(bq_model)
    
    bqml_model.uri = bq_model 
    vertex_endpoint.uri = endpoint.resource_name

#### Evaluate the Model

In [ ]:
@dsl.component(
    base_image="python:3.9",
    packages_to_install = ['pandas','db-dtypes','pyarrow','sklearn','google-cloud-bigquery']
)
def evaluate_model(
    project: str,
    bq_dataset: str,
    bq_table: str, 
    var_target: str, 
    bqml_model: dsl.Input[dsl.Model],
    metrics: dsl.Output[dsl.Metrics],
    metricsc: dsl.Output[dsl.ClassificationMetrics]
) -> NamedTuple("model_eval", [("metric", float)]):
    
    #project=PROJECT_ID
    #bq_dataset=BQ_DATASET
    #bq_table=BQ_TABLE
    #var_target=VAR_TARGET

    from google.cloud import bigquery
    import pandas as pd
    from sklearn.metrics import average_precision_score, confusion_matrix
    from collections import namedtuple
        
    # Get data from the big query table
    query = f"""
        SELECT * EXCEPT (transaction_id, splits)FROM `{project}.{BQ_DATASET}.{BQ_TABLE}` WHERE SPLITS='TEST' LIMIT 5
    """
    test_data = bq.query(query=query).to_dataframe()
    actual_class = test_data[var_target]
    test_data.drop(columns=[var_target], inplace=True)
    # predict it from the endpoint
    newobs = test_data.to_dict(orient = 'records')
    prediction = endpoint.predict(instances = newobs)
    # transform the predictions to dataframe
    preds = pd.DataFrame(prediction[0])
    preds = preds.apply(lambda col: col.apply(lambda val: val[0]), axis=0)
    preds[f'predicted_{var_target}'] = preds[f'predicted_{var_target}'].astype(int)
    # calculate preformance metrics
    avg_precision = average_precision_score(actual_class, preds[f'{var_target}_probs'], average='micro')
    conf_mat = confusion_matrix(actual_class, preds[f'predicted_{var_target}']).tolist()
    # log the perfomance metrics on the output paramenters
    metrics.log_metric('auPRC', avg_precision)
    metricsc.log_confusion_matrix(['Not Fraud', 'Fraud'], conf_mat)
    # return the metric
    model_eval = namedtuple("model_eval", ["metric"])
    return model_eval(metric = float(auPRC))

#### Train DNN

In [11]:
@dsl.component(
    base_image="python:3.9"
)
def train_dnn():
    pass

#### Compare Performance Of Models

In [9]:
@dsl.component(
    base_image="python:3.9"
)
def compare():
    pass

#### Replace Model on Endpoint

In [ ]:
@dsl.component(
    base_image="python:3.9"
)
def replace_model_on_endpoint():
    pass

#### Create Pipeline of the above functions

In [15]:
@dsl.pipeline(
     name = f'series-{SERIES}-endpoint-challenger',
     description = 'Update endpoint with challenger model (conditionally).'
)
def pipeline(
    project: str,
    region: str,
    series: str,
    bq_dataset: str,
    bq_table: str, 
    var_target: str, 
):
    # Get Current Deployed Model
    current_model = train_dnn(
    ).set_display_name('Train DNN').set_cpu_limit('1').set_caching_options(False)
    

In [13]:
@dsl.pipeline(
     name = f'series-{SERIES}-endpoint-challenger',
     description = 'Update endpoint with challenger model (conditionally).'
)
def pipeline(
    project: str,
    region: str,
    series: str,
    bq_dataset: str,
    bq_table: str, 
    var_target: str, 
):
    # Get Current Deployed Model
    current_model = get_deployed_model(
        project=project,
        region=region,
        series=series
    ).set_display_name('Get Current Model').set_caching_options(False)
    
    # Get Current Model's performance
    current_model_perfoamnce = evaluate_model(
        project=project,
        bq_dataset=bq_dataset,
        bq_table=bq_table, 
        var_target=var_target, 
        bqml_model = current_model.outputs['bqml_model']
    ).set_display_name('Evaluate Current Model').set_caching_options(False)
    
    

NameError: name 'get_deployed_model' is not defined

#### Compile the pipeline
It generates a json file that contains the infofmation about pipeline

In [16]:
# from kfp.v2 import compiler
kfp.compiler.Compiler().compile(
    pipeline_func = pipeline,
    package_path = f"{DIR}/{EXPERIMENT}.json"
)

#### Define Pipeline Job

In [17]:
parameter_values = {
    "project" : PROJECT_ID,
    "region" : REGION,
    "series": SERIES,
    "bq_dataset": BQ_DATASET,
    "bq_table": BQ_TABLE,
    "var_target": VAR_TARGET,
}



In [18]:
pipeline_job = aiplatform.PipelineJob(
    display_name = f'{EXPERIMENT}',
    template_path = f"{DIR}/{EXPERIMENT}.json",
    pipeline_root = f"{URI}/pipeline_root",
    parameter_values = parameter_values,
    enable_caching = False, # overrides all component/task settings
    labels = {'series': SERIES, 'experiment': EXPERIMENT}
)

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

#### Submit Pipeline Job


In [ ]:
SERVICE_ACCOUNT = !gcloud config list --format='value(core.account)' 
SERVICE_ACCOUNT = SERVICE_ACCOUNT[0]
SERVICE_ACCOUNT

In [ ]:
!gcloud projects get-iam-policy $PROJECT_ID --filter="bindings.members:$SERVICE_ACCOUNT" --format='table(bindings.role)' --flatten="bindings[].members"

In [ ]:
response = pipeline_job.submit(service_account=SERVICE_ACCOUNT)